In [7]:
from cobra.io import read_sbml_model
import numpy as np
import pandas as pd

model = read_sbml_model("../models/iJR904.xml.gz")

In [8]:
carbon_sources = {
    "Glucose": "EX_glc__D_e",
    "Maltose": "EX_malt_e",
    "Galactose": "EX_gal_e",
    "Glycerol": "EX_glyc_e",
    "Lactate": "EX_lac__L_e",
    "Acetate": "EX_ac_e"
}

base_medium = model.medium.copy()

# alle sechs Carbon Sources entfernen
for rxn_id in carbon_sources.values():
    base_medium.pop(rxn_id, None)

# Sauerstoff im Überschuss
base_medium["EX_o2_e"] = 999999.0

crowding_reactions = [
    rxn for rxn in model.reactions
    if rxn not in model.exchanges
    and "BIOMASS" not in rxn.id.upper()
]

crowding_constraint = model.problem.Constraint(
    0,
    ub=1.0,
    name="molecular_crowding"
)

model.add_cons_vars(crowding_constraint)
model.solver.update()

n_runs = 1000

results_fbawmc = []

#Gamma Verteilung
mean_a = 0.0040
beta = 3
rng = np.random.default_rng(seed=42)

for run in range(n_runs):
    a_values = rng.gamma(
    shape=beta,
    scale=mean_a / beta,
    size=len(crowding_reactions))

    coefficients = {}

    for rxn, a_i in zip(crowding_reactions, a_values):
        coefficients[rxn.forward_variable] = a_i
        coefficients[rxn.reverse_variable] = a_i

    crowding_constraint.set_linear_coefficients(coefficients)

    for name, exchange_id in carbon_sources.items():

        medium = base_medium.copy()
        medium[exchange_id] = 999999.0
        model.medium = medium

        solution = model.optimize()

        if solution.status == "optimal":
            growth = solution.objective_value
            uptake = abs(solution.fluxes[exchange_id])
        else:
            growth = np.nan
            uptake = np.nan

        results_fbawmc.append({
            "run": run+1,
            "carbon_source": name,
            "growth_rate": growth,
            "uptake_rate": uptake
        })
        
df_fbawmc = pd.DataFrame(results_fbawmc)

df_fbawmc.head(10)


,run,carbon_source,growth_rate,uptake_rate
0,1,Glucose,0.476117,6.802135
1,1,Maltose,0.480848,3.747926
2,1,Galactose,0.443143,6.241971
3,1,Glycerol,0.470461,10.651544
4,1,Lactate,0.317411,11.601249
5,1,Acetate,0.216160,10.704987
6,2,Glucose,0.488697,6.878554
7,2,Maltose,0.478912,3.333237
8,2,Galactose,0.407594,5.949415
9,2,Glycerol,0.515976,12.011162


In [9]:
summary_fbawmc = (
    df_fbawmc
    .groupby("carbon_source")
    .agg(
        mean_growth=("growth_rate", "mean"),
        std_growth=("growth_rate", "std"),
        mean_uptake=("uptake_rate", "mean")
    )
)

summary_fbawmc

,mean_growth,std_growth,mean_uptake
carbon_source,,,
Acetate,0.229712,0.038489,11.429016
Galactose,0.418989,0.056719,5.677108
Glucose,0.487797,0.067701,8.044538
Glycerol,0.470737,0.071485,11.010476
Lactate,0.362006,0.058382,13.337295
Maltose,0.473801,0.063638,3.424981
